# 1.4 FlashAttention 核心原理

上一节的结论：标准实现慢在**中间矩阵 S、P 反复穿越 HBM**。药方是把三个 Kernel 融合、把数据切块，让中间结果留在片上。

但有两个拦路虎：

1. **Softmax 归一化需要整行信息**（行最大值、行和），而片上一次只有一个块——看到后面块时，前面的结论可能要推翻；
2. **片上只有 256KB 量级**，整个矩阵放不下，必须切块，且切块后的结果要和整体计算**精确一致**（FlashAttention 不是近似算法！）。

FlashAttention 用两件武器解决：**Online Softmax**（对付第 1 只虎）和**分块计算 Tiling**（对付第 2 只虎）。本节是本章的理论核心。

## 一、Safe Softmax：先解决“会不会算崩”

朴素 Softmax 直接计算 $e^{x_i}$。当分数达到几百时，$e^{x_i}$ 会**溢出为 inf**（fp16 上溢阈值约 $e^{11}$，fp32 约 $e^{88}$，fp64 也只有约 $e^{709}$），结果变成 `inf / inf = nan`。真实 Attention 的分数很容易到几百，fp16 训练、fp32 推理都躲不开。

解法利用**指数平移不变性**：

$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}$$

分子分母同乘 $e^{-m}$，结果不变。取 $m$ 为**行最大值**，则所有指数 $\le 0$，永不溢出——这就是 Safe Softmax（1.2 节代码里已经默默用了它）。

用 numpy 以 **fp32** 亲手重现一次溢出，再看 Safe Softmax 如何拯救：

In [1]:
import numpy as np

# fp32 的 exp 上溢阈值约 e^88——分数几百时必然溢出，与真实训练/推理精度一致
x = np.array([500.0, 501.0, 502.0, 450.0], dtype=np.float32)

# 朴素 Softmax：直接 exp
p_naive = np.exp(x) / np.exp(x).sum()
print('朴素 Softmax 结果：', p_naive)   # 溢出：inf / inf = nan

# Safe Softmax：先减行最大值
m = x.max()
p_safe = np.exp(x - m) / np.exp(x - m).sum()
print('Safe  Softmax 结果：', p_safe)

朴素 Softmax 结果： [nan nan nan nan]
Safe  Softmax 结果： [9.0030573e-02 2.4472846e-01 6.6524094e-01 1.7364646e-23]


In [2]:
# 指数平移不变性：只要 exp(x - m) 不溢出，基准 m 换成多少结果都不变
x64 = x.astype(np.float64)              # 换回 fp64（阈值约 e^709），演示更宽的基准范围
m = x64.max()
p_ref = np.exp(x64 - m)
p_ref = p_ref / p_ref.sum()
for m_shift in [0.0, 100.0, 400.0, 502.0]:
    p = np.exp(x64 - m_shift)
    p = p / p.sum()
    print(f'基准 m = {m_shift:8.1f} → 归一化后结果一致: {np.allclose(p, p_ref)}')

# 反例：基准太小 → exp(x - m) 整体上溢，"随意换基准"在数值上失效
p_exp = np.exp(x64 - (-1000.0))
p_bad = p_exp / p_exp.sum()
print(f'基准 m = -1000.0 → exp(x-m) = {p_exp[0]}，归一化后结果一致: {np.allclose(p_bad, p_ref)}')

基准 m =      0.0 → 归一化后结果一致: True
基准 m =    100.0 → 归一化后结果一致: True
基准 m =    400.0 → 归一化后结果一致: True
基准 m =    502.0 → 归一化后结果一致: True
基准 m = -1000.0 → exp(x-m) = inf，归一化后结果一致: False


记住这个实验的结论——**只要 $e^{x_i - m}$ 不溢出（基准 m 不能太小，如上例的 -1000），归一化的“基准 m”就可以随意更换而不影响最终结果**。行最大值正是“不溢出的基准里最大的一个”，一举两得：既保住数值范围，又让指数 $\le 0$。这是 Online Softmax 全部魔法的来源：块到齐后如果发现新的最大值，把旧结果“平移修正”一下就行。

但注意：Safe Softmax 仍需要**先扫一遍整行拿到最大值、再扫一遍拿到行和**——它还是个“两遍扫描”算法，依然要求整行在手上。真正解决“分块也能算”的，是下一节的 Online Softmax。

## 二、Online Softmax：一次扫描、增量修正

### 问题设定

数据一块一块地到来（块 1、块 2、……）。任一时刻，我们只想维护三个**很小的状态**（每个 Q 行一份）：

- $m$：至今为止见过的**行最大值**；
- $l$：至今为止的**未归一化指数和**（$\sum e^{x_j - m}$ 的当前值）；
- $\tilde{O}$：至今为止的**未归一化输出累加**（$\sum P_j \cdot V_j$ 的当前值）。

目标：所有块处理完后，$O = \tilde{O} / l$ 与标准 Attention **精确相等**。

### 三步递推

![Online Softmax](images/online_softmax_update.svg)

设新块带来得分 $S_{new}$（与当前 Q 块的 M 行 × 新 KV 块的 B 列），更新规则：

① **更新行最大值**
$$m_{new} = \max\left(m_{old},\; \text{rowmax}(S_{new})\right)$$

② **算新块的概率**（对照新的基准）
$$\tilde{P} = e^{S_{new} - m_{new}}$$

③ **重标定旧状态**（关键一步！）
$$\alpha = e^{m_{old} - m_{new}} \;\le\; 1 \qquad \text{（旧基准到新基准的修正因子）}$$
$$l_{new} = \alpha \, l_{old} + \text{rowsum}(\tilde{P}) \qquad \tilde{O}_{new} = \alpha \, \tilde{O}_{old} + \tilde{P} \, V_{new}$$

收尾：$O = \tilde{O} / l$。

### 为什么正确？

平移不变性：旧的 $l_{old}, \tilde{O}_{old}$ 是以 $m_{old}$ 为基准累加的；换到新基准 $m_{new}$，只需全部乘上 $e^{m_{old} - m_{new}}$——这正是 $\alpha$。于是**每个块处理完，三个状态都等价于“用当前基准算出的部分和”**。全部块结束、基准不再变化时，除以最终的 $l$ 就得到精确的 Softmax 结果。

> 通俗版：记账时“单位”换了（最大值更新了），把旧账全部按汇率 $\alpha$ 折算一遍，账本永远有效。

注意每来一个块，$\alpha \le 1$ 意味着旧结果**只会被缩小**，数值上天然稳定（这就是 1.5 节数值实验要验证的）。

## 三、分块计算（Tiling）：外层 Q 块，内层 KV 块

Online Softmax 解决了“整行才能归一化”的障碍，分块就是它的用武之地：

![FlashAttention 分块](images/flash_tiling.svg)

### 伪代码（单头、无掩码版）

```text
把 Q 切成 Tn = n / M 个块（每块 M 行），K/V 切成 n / B 个块（每块 B 行）
for 每个 Q 块 Qi (M 行)：                 # 外层：每块 Q 只从 HBM 读一次
    m = -inf, l = 0, Oacc = 0             # 三个状态，驻留片上
    for 每个 KV 块 (Kj, Vj) (B 行)：      # 内层：小块搬入片上
        S    = Qi · Kjᵀ / √d              # GEMM ①   (M × B)
        按 1.4 二节三步递推更新：
            m_new, α, P̃ = …
            l    = α·l + rowsum(P̃)
            Oacc = α·Oacc + P̃ · Vj        # GEMM ②   (M × B) × (B × d)
    O[Qi] = Oacc / l                      # 收尾：一次性写回 HBM
```

### 逐条对照性能收益

| 设计 | 消灭的开销 | 对应 1.3 节的哪笔账 |
|--|--|--|
S、P 只以 `M × B` 小块存在，用完即弃 | 5 次 `n²` 量级的 HBM 读写 | S/P 写下+读回 |
Q 块驻留片上，被所有 KV 块复用 | Q 只读 1 次 | Q 读取 |
每个 KV 块被 `n/M` 个 Q 块复用（片上容量能缓存时） | KV 读取摊薄 | K/V 读取 |
O 按 Q 块粒度流式写出 | O 只写 1 次 | O 写出 |
两个 GEMM 与 Softmax 交织执行 | Kernel 间切换、中间落盘 | 三 Kernel 拆分 |

### 与 GEMM 分块的类比

如果你学过矩阵乘的 Tile 优化，会发现 FlashAttention 的循环结构与 GEMM 完全同构：**外层分块让“驻留方”复用，内层分块让“流过方”小块搬运**。区别在于 GEMM 分块只有累加（$C += A_i \cdot B_j$，天然可结合），而 Attention 中间隔着 Softmax——不可结合，所以必须引入 $m, l, \alpha$ 这套“重标定”机制。**FlashAttention 的本质：把不可结合的 Attention 改写成可增量累加的形式。**

## 四、因果掩码：整块跳过

生成式模型推理时通常开启因果掩码（causal mask）：第 `i` 个 token 只能看到位置 `j ≤ i` 的键，否则就是“偷看未来”。元素级实现是给 `S[i, j> i]` 填 $-\infty$（exp 后变 0）。

分块之后有一个漂亮的副产品——**掩码可以按块决策**：

![因果掩码分块](images/causal_mask_blocks.svg)

- 若 Q 块 `i` 与 KV 块 `j` **完全不重叠且 j ≤ i**（图中绿色区）：整块保留，无需任何掩码；
- 若 **j > i**（灰色区）：整块置 $-\infty$——**干脆不进内层循环**，计算和访存一起省掉；
- 只有**对角块**（黄色区）：块内既有过去也有未来，才需要逐元素掩码。

于是掩码从“逐元素 if”升级为“整块 skip”，长序列下大约省去一半计算（对角块占比很小）。代码上仅仅是内层循环的上界变化：

```text
for j in range(0, block_index_of(i) + 1):   # 只需遍历到 i 所在的块
    if j < block_index_of(i):  无需掩码
    else:                      对角块，块内掩码 j_pos > i_pos → -inf
```

> 小知识：在 KV Cache 增量推理（Decode）场景里，新 token 的 Q 天然排在所有历史 K 之后，**因果约束自动满足**，连掩码都不用加——这也是 Decode 阶段比 Prefill 阶段“轻”的原因之一。

## 五、总结：FlashAttention 到底快在哪

| 维度 | 标准实现 | FlashAttention |
|--|--|--|
HBM 访存（单头，fp16） | $8nd + 10n^2$ 字节量级 | $\approx 8nd + \frac{4d}{M}n^2$（n² 系数从 10 降到 4d/M，重读可被 L2 拦截） |
中间矩阵 | S、P 落 HBM | 只存在片上，用完即弃 |
额外显存 | $O(n^2)$（训练时还要存 P 反传） | $O(n)$ |
数值结果 | — | **与标准实现精确一致**（不是近似！） |

一句话总结：**FlashAttention 是 IO-aware 的精确 Attention——计算量几乎不变，通过重构数据流把 $n^2$ 量级的显存流量压掉，让算子从“搬运工”变回“计算者”。**

## 六、从数学到昇腾算子：原理如何落到 `FusedInferAttentionScore`

昇腾 NPU 上的 `FusedInferAttentionScore` 算子，正是本章原理的工程化。提前建立映射关系：

| 本章概念 | 昇腾实现中的对应 |
|--|--|
外层 Q 块 M、内层 KV 块 B | Tiling 参数 `blockM / blockN`（按硬件 Cube/Vector 规格选取） |
GEMM ①、GEMM ② | AI Core 的 **Cube 单元**（矩阵乘） |
Online Softmax 三步递推 | **Vector 单元** + 片上 Buffer（UB）上的 exp/max/广播乘 |
状态 m、l、Oacc 驻留 | Unified Buffer 中跨块驻留的寄存器/Buffer |
KV 分块 + 因果块跳过 | 内层循环按 block table 遍历、上界裁剪 |
KV Cache、GQA/MQA（1.2 节） | 算子入参 `key/value`（缓存）、`num_heads / num_kv_heads`、`block_table` |
全量 / 增量场景（1.2 节 KV Cache 图） | 同一算子的 **Prompt / Decode** 两条路径 |

看一眼真实算子接口里的几个“眼熟”参数（节选自 ops-transformer 的 `FusedInferAttentionScore` 规格）：

- `query` / `key` / `value`：1.2 节的 Q、K（V），其中 key/value 即 KV Cache；
- `num_heads` / `num_kv_heads`：两者不等即为 GQA/MQA；
- `blockTable` + `blockSize`：KV Cache 在显存中的离散分块表——1.2 节图中“逐格填入”的格子物理上就散在这些 block 里；
- `attenMask`：1.4 节第四节的掩码（支持块级裁剪）；
- `inputLayout`（BSH/BSND/SBND/BNSD）：就是 1.2 节 n×d 矩阵在显存里的不同摆放方式。

是不是突然觉得这个复杂接口没那么吓人了？——它只是把本章每一张图变成了参数。

## 章节测验

1. Safe Softmax 为什么减行最大值？为什么减最大值后一定不溢出？
2. 指数平移不变性如何保证“换基准”不影响结果？写出 $\alpha$ 的表达式并解释它为什么 $\le 1$。

3. 填空：Online Softmax 三步递推中，$l_{new} = \underline{\qquad\quad}$，$\tilde{O}_{new} = \underline{\qquad\quad}$。
4. FlashAttention 的外层循环切谁、内层循环切谁？为什么这么分工（提示：谁驻留、谁流过）？
5. 因果掩码下，内层循环对第 `i` 个 Q 块要遍历到哪个 KV 块为止？哪一类块不需要逐元素掩码？
6. （判断）FlashAttention 是一种近似算法，牺牲精度换速度。（对/错，为什么）

> 答案见 `answer/01.04_answer.txt`。

下一节：[1.5 Online Softmax 数值实验](01.05_online_softmax_experiment.ipynb)——亲手把 1.4 节的公式写成代码，验证它与标准 Attention 逐位一致。